In [1]:
# 🔹 Standard imports
import os
import shutil
import numpy as np
from collections import Counter

# 🔹 TensorFlow / Keras
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.applications.efficientnet import preprocess_input
from tensorflow.keras.layers import Dense, Dropout, GlobalAveragePooling2D
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

# 🔹 Sklearn
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import confusion_matrix, classification_report

print(" Imports done")


2026-01-16 20:16:42.164625: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-01-16 20:16:42.201912: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-01-16 20:16:47.220610: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


 Imports done


In [5]:
# 🔹 Paths
dataset_dir = "/home/karan/Datasets/FINAL_UNIFIED_DATASET"
split_dir   = "/home/karan/Datasets/FINAL_SPLIT_DATASET"

# 🔹 Parameters
img_size = (224, 224)
batch_size = 32
initial_epochs = 5
finetune_epochs = 15

print(" Paths & parameters set")


 Paths & parameters set


In [6]:
def clean_class_name(cls):
    cls = cls.replace(" ", "_")
    cls = cls.replace("(", "").replace(")", "")
    cls = cls.replace("-", "_")
    cls = cls.replace(",", "_")
    return cls

classes = sorted([d for d in os.listdir(dataset_dir) 
                  if os.path.isdir(os.path.join(dataset_dir, d))])

for cls in classes:
    clean_name = clean_class_name(cls)
    if cls != clean_name:
        os.rename(os.path.join(dataset_dir, cls),
                  os.path.join(dataset_dir, clean_name))

classes = sorted([d for d in os.listdir(dataset_dir) 
                  if os.path.isdir(os.path.join(dataset_dir, d))])

print(f" Total classes: {len(classes)}")


 Total classes: 54


In [7]:
os.makedirs(split_dir, exist_ok=True)

for split in ["train", "val", "test"]:
    os.makedirs(os.path.join(split_dir, split), exist_ok=True)

for cls in classes:
    cls_path = os.path.join(dataset_dir, cls)
    images = os.listdir(cls_path)

    train_imgs, temp_imgs = train_test_split(
        images, test_size=0.2, random_state=42
    )
    val_imgs, test_imgs = train_test_split(
        temp_imgs, test_size=0.5, random_state=42
    )

    for split_name, split_imgs in zip(
        ["train", "val", "test"],
        [train_imgs, val_imgs, test_imgs]
    ):
        split_cls_dir = os.path.join(split_dir, split_name, cls)
        os.makedirs(split_cls_dir, exist_ok=True)

        for img in split_imgs:
            shutil.copy2(
                os.path.join(cls_path, img),
                os.path.join(split_cls_dir, img)
            )

print(" Dataset split completed")


 Dataset split completed


In [8]:
train_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input,
    rotation_range=15,
    width_shift_range=0.1,
    height_shift_range=0.1,
    zoom_range=0.1,
    horizontal_flip=True
)

val_datagen  = ImageDataGenerator(preprocessing_function=preprocess_input)
test_datagen = ImageDataGenerator(preprocessing_function=preprocess_input)

train_gen = train_datagen.flow_from_directory(
    os.path.join(split_dir, "train"),
    target_size=img_size,
    batch_size=batch_size,
    class_mode="categorical"
)

val_gen = val_datagen.flow_from_directory(
    os.path.join(split_dir, "val"),
    target_size=img_size,
    batch_size=batch_size,
    class_mode="categorical"
)

test_gen = test_datagen.flow_from_directory(
    os.path.join(split_dir, "test"),
    target_size=img_size,
    batch_size=batch_size,
    class_mode="categorical",
    shuffle=False
)

print(" Data generators ready")


Found 54273 images belonging to 54 classes.
Found 6783 images belonging to 54 classes.
Found 6812 images belonging to 54 classes.
 Data generators ready


In [9]:
train_gen.class_indices


{'Apple___Apple_scab': 0,
 'Apple___Black_rot': 1,
 'Apple___Cedar_apple_rust': 2,
 'Apple___healthy': 3,
 'Blueberry___healthy': 4,
 'Cherry_including_sour___Powdery_mildew': 5,
 'Cherry_including_sour___healthy': 6,
 'Corn_Blight': 7,
 'Corn_Common_Rust': 8,
 'Corn_Greay_Leaf_Spot': 9,
 'Corn_Healthy': 10,
 'Corn_maize___Cercospora_leaf_spot_Gray_leaf_spot': 11,
 'Corn_maize___Common_rust_': 12,
 'Corn_maize___Northern_Leaf_Blight': 13,
 'Corn_maize___healthy': 14,
 'Grape___Black_rot': 15,
 'Grape___Esca_Black_Measles': 16,
 'Grape___Leaf_blight_Isariopsis_Leaf_Spot': 17,
 'Grape___healthy': 18,
 'Orange___Haunglongbing_Citrus_greening': 19,
 'Peach___Bacterial_spot': 20,
 'Peach___healthy': 21,
 'Pepper__bell___Bacterial_spot': 22,
 'Pepper__bell___healthy': 23,
 'Potato___Early_blight': 24,
 'Potato___Late_blight': 25,
 'Potato___healthy': 26,
 'Raspberry___healthy': 27,
 'Rice_Bacterial_Blight': 28,
 'Rice_Blast': 29,
 'Rice_Brown_Spot': 30,
 'Sorghum_Anthracnose_Red_Rot': 31,
 '

In [10]:
import json

with open("class_indices.json", "w") as f:
    json.dump(train_gen.class_indices, f)


In [ ]:
y_train = train_gen.classes

class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.unique(y_train),
    y=y_train
)

class_weights_dict = dict(enumerate(class_weights))
print(" Class weights computed")


In [ ]:
base_model = EfficientNetB0(
    weights="imagenet",
    include_top=False,
    input_shape=(224, 224, 3)
)

base_model.trainable = False

x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dropout(0.3)(x)
outputs = Dense(len(classes), activation="softmax")(x)

model = Model(inputs=base_model.input, outputs=outputs)

model.compile(
    optimizer=Adam(1e-3),
    loss=tf.keras.losses.CategoricalCrossentropy(label_smoothing=0.1),
    metrics=["accuracy"]
)

model.summary()


In [ ]:
history = model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=initial_epochs,
    class_weight=class_weights_dict
)


In [ ]:
base_model.trainable = True

for layer in base_model.layers[:-50]:
    layer.trainable = False

model.compile(
    optimizer=Adam(1e-5),
    loss=tf.keras.losses.CategoricalCrossentropy(label_smoothing=0.1),
    metrics=["accuracy"]
)

callbacks = [
    EarlyStopping(
        monitor="val_loss",
        patience=3,
        restore_best_weights=True
    ),
    ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.3,
        patience=2,
        min_lr=1e-6
    )
]

history_finetune = model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=finetune_epochs,
    callbacks=callbacks,
    class_weight=class_weights_dict
)


In [ ]:
test_loss, test_acc = model.evaluate(test_gen)
print(f"✅ Test Accuracy: {test_acc*100:.2f}%")


In [ ]:
y_pred = np.argmax(model.predict(test_gen), axis=1)
y_true = test_gen.classes

print(classification_report(y_true, y_pred))


In [ ]:
model.save("plant_disease_effnetb0.keras")
model.save("plant_disease_effnetb0.h5")

print("✅ Model saved successfully")
